# 📊 Análise de Sentimento - Unidade E

## Objetivo
Analisar a percepção dos pacientes a partir das avaliações públicas.

## Metodologia
- NLP (análise de sentimento)
- análise temporal
- análise de frequência de palavras

In [1]:
import pandas as pd
import numpy as np

In [3]:
from transformers import pipeline

In [4]:
import re
from collections import Counter

In [5]:
df_eco = pd.read_csv("02_EcoMedical_Google_Reviews.csv")
df_eco.head()

,unidade,autor,rating,data,comentario
0,Eco Medical,Paulo Pontes,2,2026,Cheguei às 20h30 e não fui atendido. Eles ence...
1,Eco Medical,Flávia Quaresma,5,2025,Avaliação sem texto
2,Eco Medical,Eli Nascimento,3,2025,O atendimento não é dos melhores e as recepcio...
3,Eco Medical,Débora Hellen,5,2025,A Dra. Lorena foi a única que me deixou tranqu...
4,Eco Medical,Alex Marinho,1,2025,Já é a segunda vez que chego lá e não sou aten...


In [6]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

In [7]:
df_eco["comentario"] = df_eco["comentario"].fillna("Avaliação sem texto")

df_sentimento = df_eco[df_eco["comentario"] != "Avaliação sem texto"].copy()

len(df_sentimento)

110

In [8]:
df_eco["rating"].value_counts().sort_index()

,count
rating,
1,45
2,5
3,3
4,6
5,99


In [9]:
df_eco["rating"].value_counts().sort_index()

,count
rating,
1,45
2,5
3,3
4,6
5,99


In [10]:
df_eco["rating"].value_counts(normalize=True).sort_index() * 100

,proportion
rating,
1,28.481013
2,3.164557
3,1.898734
4,3.797468
5,62.658228


In [11]:
df_eco["data"].value_counts().sort_index()

,count
data,
2016,5
2017,10
2018,8
2019,16
2020,22
2021,18
2022,19
2023,21
2024,16


In [12]:
df_eco.groupby("data")["rating"].mean().round(2)

,rating
data,
2016,5.00
2017,5.00
2018,4.50
2019,3.81
2020,4.27
2021,4.56
2022,3.84
2023,2.67
2024,3.19


In [13]:
df_critico = df_sentimento[df_sentimento["data"].isin([2023, 2024, 2025, 2026])].copy()

len(df_critico)

51

In [17]:
def analisar_sentimento(texto):
    resultado = sentiment_pipeline(
        str(texto),
        truncation=True,
        max_length=128
    )[0]
    return resultado["label"]

In [18]:
df_critico["sentimento"] = df_critico["comentario"].apply(analisar_sentimento)

df_critico["sentimento"].value_counts(normalize=True) * 100

,proportion
sentimento,
NEG,62.745098
POS,29.411765
NEU,7.843137


In [22]:
import re
from collections import Counter

stopwords = [
    "de","da","do","das","dos","a","o","as","os","e","é","em","para",
    "com","que","não","na","no","uma","um","mais","foi","por","se",
    "ao","à","às","como","já","eu","ele","ela","me","minha","meu",
    "muito","porque","isso","está","estava","são","ser","tem","pra",
    "pelo","pela","até","fui","era"
]

def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    palavras = texto.split()
    palavras = [p for p in palavras if p not in stopwords and len(p) > 2]
    return palavras

In [23]:
df_neg_critico = df_critico[df_critico["sentimento"] == "NEG"].copy()

todas_palavras = []

for comentario in df_neg_critico["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras.extend(palavras)

Counter(todas_palavras).most_common(20)

[('atendimento', 25),
 ('unidade', 11),
 ('horário', 9),
 ('clínica', 9),
 ('filho', 9),
 ('fazer', 8),
 ('exame', 8),
 ('paciente', 8),
 ('atendente', 8),
 ('atendido', 7),
 ('hora', 7),
 ('consulta', 7),
 ('horas', 7),
 ('espera', 7),
 ('whatsapp', 7),
 ('exames', 7),
 ('falta', 7),
 ('atender', 7),
 ('esse', 6),
 ('mas', 6)]

In [ ]:
df_neg_critico = df_critico[df_critico["sentimento"] == "NEG"].copy()

todas_palavras = []

for comentario in df_neg_critico["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras.extend(palavras)

Counter(todas_palavras).most_common(20)

[('atendimento', 12),
 ('profissionais', 6),
 ('excelente', 5),
 ('ótimo', 4),
 ('bem', 3),
 ('médico', 3),
 ('tanto', 3),
 ('quanto', 3),
 ('clínica', 3),
 ('única', 2),
 ('ótima', 2),
 ('vezes', 2),
 ('urgência', 2),
 ('atencioso', 2),
 ('rápido', 2),
 ('bastante', 2),
 ('gosta', 2),
 ('profissional', 2),
 ('parte', 2),
 ('carro', 2)]